# Notebook 13B — Final Reconciliation and Submission Assets

## Grain AI–MDPI

This notebook resolves the final provenance inconsistency identified after Notebook 13. It does **not** train or select models.

### Frozen primary models and authoritative sources

| Analysis layer | Dry Bean | INIAP Rice | Authoritative source |
|---|---|---|---|
| Primary performance, calibration, confusion and classwise metrics | R1 full native + RBF-SVM | R4 hybrid log-area + RBF-SVM | Notebook 05 |
| Selective classification | Same frozen champions | Same frozen champions | Notebook 07 |
| Canonical conformal prediction | Same frozen champions | Same frozen champions | Notebook 08 v2 |
| Corrected representation contrasts, equivalence, controlled R1 importance, scale sensitivity and leakage audits | Confirmatory reanalysis | Confirmatory reanalysis | Notebook 12 v2 |

Notebook 12 refitted models are **not** used for the primary performance, calibration, selective-classification or conformal tables.

### Required input ZIP files

- `NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip`
- `NOTEBOOK07_SELECTIVE_UNCERTAINTY_OUTPUTS.zip`
- `NOTEBOOK08_V2_CANONICAL_CONFORMAL_OUTPUTS.zip`
- `NOTEBOOK12_REVIEWER_DRIVEN_REANALYSIS_OUTPUTS_V2.zip`

### Output

The final ZIP is:

`NOTEBOOK13B_RECONCILED_FINAL_MANUSCRIPT_ASSETS.zip`

All scale-dependent measurements are interpreted in image space: pixels for linear descriptors, squared pixels for area descriptors, and dimensionless units for ratios and shape factors.

In [ ]:
# ============================================================
# 1. Dependencies and configuration
# ============================================================
import importlib
import importlib.metadata
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pyarrow": "pyarrow",
    "tabulate": "tabulate",
}

packages_to_install = []

for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        packages_to_install.append(package_name)

if packages_to_install:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *packages_to_install,
        ]
    )

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import io
import json
import math
import os
import platform
import shutil
import tempfile
import time
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
)

warnings.filterwarnings("ignore")

SOURCE_SPECS = {
    "notebook05": {
        "filename": "NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip",
        "signature": "03_protocol/baseline_calibration_protocol_v1.json",
    },
    "notebook07": {
        "filename": "NOTEBOOK07_SELECTIVE_UNCERTAINTY_OUTPUTS.zip",
        "signature": "03_protocol/selective_uncertainty_protocol_v1.json",
    },
    "notebook08": {
        "filename": "NOTEBOOK08_V2_CANONICAL_CONFORMAL_OUTPUTS.zip",
        "signature": "03_protocol/canonical_conformal_prediction_protocol_v2.json",
    },
    "notebook12": {
        "filename": "NOTEBOOK12_REVIEWER_DRIVEN_REANALYSIS_OUTPUTS_V2.zip",
        "signature": "03_protocol/reviewer_reanalysis_protocol_v2.json",
    },
}

FROZEN_CHAMPIONS = {
    "dry_bean": {
        "representation": "R1_full_native",
        "algorithm": "rbf_svm",
        "candidate_id": "rbf_svm_C02",
    },
    "iniap_rice": {
        "representation": "R4_hybrid_log_area",
        "algorithm": "rbf_svm",
        "candidate_id": "rbf_svm_C04",
    },
}

N_BOOTSTRAP = 2000
CONFIDENCE_LEVEL = 0.95
BOOTSTRAP_SEED = 2026081302
N_RELIABILITY_BINS = 15

if Path("/content").exists():
    LOCAL_ROOT = Path(
        "/content/Grain_AI_MDPI_notebook13B"
    )
else:
    LOCAL_ROOT = Path(
        "/mnt/data/notebook13B_local_run"
    )

INPUT_DIR = LOCAL_ROOT / "00_input"
EXTRACT_ROOT = LOCAL_ROOT / "01_extracted"
OUTPUT_ROOT = LOCAL_ROOT / "02_outputs"

if OUTPUT_ROOT.exists():
    shutil.rmtree(
        OUTPUT_ROOT
    )

PROTOCOL_DIR = OUTPUT_ROOT / "03_protocol"
RESULTS_DIR = OUTPUT_ROOT / "07_results" / "13B_FINAL_RECONCILIATION"
FIGURES_MAIN_DIR = OUTPUT_ROOT / "08_figures" / "main"
FIGURES_SUPP_DIR = OUTPUT_ROOT / "08_figures" / "supplementary"
TABLES_MAIN_DIR = OUTPUT_ROOT / "09_tables" / "main"
TABLES_SUPP_DIR = OUTPUT_ROOT / "09_tables" / "supplementary"
CLAIMS_DIR = OUTPUT_ROOT / "10_claim_register"

for directory in [
    INPUT_DIR,
    EXTRACT_ROOT,
    PROTOCOL_DIR,
    RESULTS_DIR,
    FIGURES_MAIN_DIR,
    FIGURES_SUPP_DIR,
    TABLES_MAIN_DIR,
    TABLES_SUPP_DIR,
    CLAIMS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print(
    "scikit-learn:",
    importlib.metadata.version("scikit-learn"),
)
print(
    "pyarrow:",
    importlib.metadata.version("pyarrow"),
)
print("Platform:", platform.platform())
print("CPU count:", os.cpu_count())

In [ ]:
# ============================================================
# 2. Robust source discovery: Colab Drive or local /mnt/data
# ============================================================
IN_COLAB = False
COLAB_FILES = None
MOUNT_ROOT = None
DRIVE_PROJECT_ROOT = None

try:
    from google.colab import drive, files

    IN_COLAB = True
    COLAB_FILES = files

    existing_mounts = [
        Path("/content/drive"),
        Path("/content/drive_nb12"),
        Path("/content/drive_nb13"),
        Path("/content/drive_nb13b"),
    ]

    for candidate in existing_mounts:
        if (
            candidate.exists()
            and (
                candidate
                / "MyDrive"
            ).exists()
        ):
            MOUNT_ROOT = candidate
            break

    if MOUNT_ROOT is None:
        mount_point = Path(
            "/content/drive_nb13b"
        )

        if mount_point.exists():
            if any(
                mount_point.iterdir()
            ):
                shutil.rmtree(
                    mount_point
                )

        mount_point.mkdir(
            parents=True,
            exist_ok=True,
        )

        drive.mount(
            str(
                mount_point
            ),
            force_remount=False,
        )

        MOUNT_ROOT = mount_point

    DRIVE_PROJECT_ROOT = (
        MOUNT_ROOT
        / "MyDrive"
        / "Grain_Robustness_Q1"
    )

    DRIVE_PROJECT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Google Drive project root:",
        DRIVE_PROJECT_ROOT,
    )

except ModuleNotFoundError:
    print(
        "Hosted Colab was not detected. "
        "Local source discovery will be used."
    )

SEARCH_ROOTS = [
    Path("/content"),
    Path("/mnt/data"),
]

if DRIVE_PROJECT_ROOT is not None:
    SEARCH_ROOTS.insert(
        0,
        DRIVE_PROJECT_ROOT,
    )

def zip_contains_signature(
    path,
    signature,
):
    try:
        with zipfile.ZipFile(
            path,
            "r",
        ) as archive:
            return (
                signature
                in archive.namelist()
            )
    except Exception:
        return False

def locate_source(
    filename,
    signature,
):
    exact_candidates = [
        root
        / filename
        for root in SEARCH_ROOTS
    ]

    for candidate in exact_candidates:
        if (
            candidate.exists()
            and zip_contains_signature(
                candidate,
                signature,
            )
        ):
            return candidate

    stem = Path(
        filename
    ).stem

    candidates = []

    for root in SEARCH_ROOTS:
        if not root.exists():
            continue

        candidates.extend(
            root.glob(
                f"{stem}*.zip"
            )
        )

    valid_candidates = [
        candidate
        for candidate in candidates
        if zip_contains_signature(
            candidate,
            signature,
        )
    ]

    if valid_candidates:
        return max(
            valid_candidates,
            key=lambda path: path.stat().st_mtime,
        )

    if (
        IN_COLAB
        and COLAB_FILES is not None
    ):
        print(
            "Upload:",
            filename,
        )

        uploaded = COLAB_FILES.upload()

        matching_names = [
            name
            for name in uploaded
            if zip_contains_signature(
                Path("/content")
                / name,
                signature,
            )
        ]

        if len(
            matching_names
        ) != 1:
            raise FileNotFoundError(
                f"A valid {filename} was not uploaded."
            )

        return (
            Path("/content")
            / matching_names[
                0
            ]
        )

    raise FileNotFoundError(
        filename
    )

source_paths = {}

for source_key, source_spec in SOURCE_SPECS.items():
    source_paths[
        source_key
    ] = locate_source(
        source_spec[
            "filename"
        ],
        source_spec[
            "signature"
        ],
    )

source_inventory = pd.DataFrame(
    [
        {
            "source": source_key,
            "filename": SOURCE_SPECS[
                source_key
            ][
                "filename"
            ],
            "located_path": str(
                source_path
            ),
            "size_bytes": source_path.stat().st_size,
        }
        for source_key, source_path
        in source_paths.items()
    ]
)

display(source_inventory)

In [ ]:
# ============================================================
# 3. Extraction, source validation and SHA-256 manifest checks
# ============================================================
def as_boolean_series(
    values,
):
    if pd.api.types.is_bool_dtype(
        values
    ):
        return values.astype(bool)

    normalized = (
        values.astype(str)
        .str.strip()
        .str.lower()
    )

    valid = normalized.isin(
        [
            "true",
            "false",
            "1",
            "0",
        ]
    )

    if not valid.all():
        raise ValueError(
            "A boolean column contains unexpected values."
        )

    return normalized.isin(
        [
            "true",
            "1",
        ]
    )

def sha256_file(
    path,
):
    digest = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as stream:
        for block in iter(
            lambda: stream.read(
                1024
                * 1024
            ),
            b"",
        ):
            digest.update(
                block
            )

    return digest.hexdigest()

source_roots = {}

for source_key, source_path in source_paths.items():
    destination = (
        EXTRACT_ROOT
        / source_key
    )

    if destination.exists():
        shutil.rmtree(
            destination
        )

    destination.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        source_path,
        "r",
    ) as archive:
        corrupt_member = archive.testzip()

        if corrupt_member is not None:
            raise RuntimeError(
                f"Corrupt ZIP member in {source_key}: "
                f"{corrupt_member}"
            )

        archive.extractall(
            destination
        )

    source_roots[
        source_key
    ] = destination

def load_csv(
    source_key,
    relative_path,
):
    path = (
        source_roots[
            source_key
        ]
        / relative_path
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    return pd.read_csv(
        path
    )

def load_json(
    source_key,
    relative_path,
):
    path = (
        source_roots[
            source_key
        ]
        / relative_path
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

validation_specs = {
    "notebook05": (
        "07_results/05_BASELINE_CALIBRATION/"
        "07_final_baseline_validation.csv"
    ),
    "notebook07": (
        "07_results/07_SELECTIVE_UNCERTAINTY/"
        "05_final_selective_validation.csv"
    ),
    "notebook08": (
        "07_results/08_V2_CANONICAL_CONFORMAL/"
        "05_final_conformal_validation_v2.csv"
    ),
    "notebook12": (
        "07_results/12_REVIEWER_REANALYSIS/"
        "03_final_reviewer_reanalysis_validation.csv"
    ),
}

manifest_specs = {
    "notebook05": (
        "07_results/05_BASELINE_CALIBRATION/"
        "08_output_integrity_manifest.csv"
    ),
    "notebook07": (
        "07_results/07_SELECTIVE_UNCERTAINTY/"
        "06_output_integrity_manifest.csv"
    ),
    "notebook08": (
        "07_results/08_V2_CANONICAL_CONFORMAL/"
        "06_output_integrity_manifest_v2.csv"
    ),
    "notebook12": (
        "07_results/12_REVIEWER_REANALYSIS/"
        "04_output_integrity_manifest.csv"
    ),
}

source_validation_rows = []
manifest_validation_rows = []

for source_key in SOURCE_SPECS:
    validation = load_csv(
        source_key,
        validation_specs[
            source_key
        ],
    )

    passed = bool(
        as_boolean_series(
            validation[
                "passed"
            ]
        ).all()
    )

    source_validation_rows.append(
        {
            "source": source_key,
            "n_checks": len(
                validation
            ),
            "all_checks_passed": passed,
        }
    )

    if not passed:
        display(
            validation.loc[
                ~as_boolean_series(
                    validation[
                        "passed"
                    ]
                )
            ]
        )

        raise RuntimeError(
            f"{source_key} validation failed."
        )

    manifest = load_csv(
        source_key,
        manifest_specs[
            source_key
        ],
    )

    for _, row in manifest.iterrows():
        file_path = (
            source_roots[
                source_key
            ]
            / str(
                row[
                    "relative_path"
                ]
            )
        )

        exists = file_path.exists()

        size_matches = (
            exists
            and file_path.stat().st_size
            == int(
                row[
                    "size_bytes"
                ]
            )
        )

        hash_matches = (
            exists
            and sha256_file(
                file_path
            )
            == str(
                row[
                    "sha256"
                ]
            )
        )

        manifest_validation_rows.append(
            {
                "source": source_key,
                "relative_path": str(
                    row[
                        "relative_path"
                    ]
                ),
                "exists": exists,
                "size_matches": size_matches,
                "hash_matches": hash_matches,
                "passed": bool(
                    exists
                    and size_matches
                    and hash_matches
                ),
            }
        )

source_validation = pd.DataFrame(
    source_validation_rows
)

source_manifest_validation = pd.DataFrame(
    manifest_validation_rows
)

if not source_manifest_validation[
    "passed"
].all():
    display(
        source_manifest_validation.loc[
            ~source_manifest_validation[
                "passed"
            ]
        ]
    )

    raise RuntimeError(
        "At least one source manifest did not validate."
    )

display(source_validation)

print(
    "Manifest entries validated:",
    len(
        source_manifest_validation
    ),
)

In [ ]:
# ============================================================
# 4. Load exact frozen champion predictions and metadata
# ============================================================
NB05_PREDICTION_ROOT = (
    source_roots[
        "notebook05"
    ]
    / "06_predictions"
    / "05_BASELINE"
)

NB05_MODEL_ROOT = (
    source_roots[
        "notebook05"
    ]
    / "05_models"
)

def champion_metadata(
    dataset_id,
):
    champion = FROZEN_CHAMPIONS[
        dataset_id
    ]

    path = (
        NB05_MODEL_ROOT
        / dataset_id
        / champion[
            "representation"
        ]
        / (
            f"{champion['algorithm']}_metadata.json"
        )
    )

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

def champion_prediction_frame(
    dataset_id,
    partition,
):
    champion = FROZEN_CHAMPIONS[
        dataset_id
    ]

    path = (
        NB05_PREDICTION_ROOT
        / dataset_id
        / champion[
            "representation"
        ]
        / (
            f"{champion['algorithm']}_"
            f"{partition}_predictions.parquet"
        )
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    frame = pd.read_parquet(
        path
    )

    for column in [
        "record_id",
        "duplicate_group_id",
        "class_label",
        "partition",
    ]:
        frame[
            column
        ] = frame[
            column
        ].astype(str)

    return frame

champion_metadata_map = {
    dataset_id: champion_metadata(
        dataset_id
    )
    for dataset_id in FROZEN_CHAMPIONS
}

prediction_frames = {
    (
        dataset_id,
        partition,
    ): champion_prediction_frame(
        dataset_id,
        partition,
    )
    for dataset_id in FROZEN_CHAMPIONS
    for partition in [
        "calibration",
        "test",
    ]
}

frozen_model_catalog = pd.DataFrame(
    [
        {
            "dataset_id": dataset_id,
            "representation": metadata[
                "representation"
            ],
            "algorithm": metadata[
                "algorithm"
            ],
            "candidate_id": metadata[
                "candidate_id"
            ],
            "temperature": metadata[
                "temperature"
            ],
            "train_records": metadata[
                "train_records"
            ],
            "calibration_records": metadata[
                "calibration_records"
            ],
            "test_records": metadata[
                "test_records"
            ],
            "feature_count": len(
                metadata[
                    "feature_columns"
                ]
            ),
            "features": " | ".join(
                metadata[
                    "feature_columns"
                ]
            ),
            "class_order": " | ".join(
                metadata[
                    "class_order"
                ]
            ),
            "source": (
                "Notebook 05 exact frozen model"
            ),
        }
        for dataset_id, metadata
        in champion_metadata_map.items()
    ]
)

display(frozen_model_catalog)

In [ ]:
# ============================================================
# 5. Probability, metric, calibration and bootstrap functions
# ============================================================
def ordered_columns(
    frame,
    prefix,
):
    columns = [
        column
        for column in frame.columns
        if column.startswith(
            prefix
        )
    ]

    return sorted(
        columns,
        key=lambda column: int(
            column.rsplit(
                "_",
                1,
            )[
                -1
            ]
        ),
    )

def probability_matrix(
    frame,
    status,
):
    prefix = {
        "native": "prob_native_",
        "temperature_scaled": (
            "prob_calibrated_"
        ),
    }[
        status
    ]

    columns = ordered_columns(
        frame,
        prefix,
    )

    if not columns:
        raise RuntimeError(
            f"No probability columns for {status}."
        )

    probabilities = frame[
        columns
    ].to_numpy(
        dtype=float
    )

    row_sums = probabilities.sum(
        axis=1,
        keepdims=True,
    )

    if (
        not np.isfinite(
            probabilities
        ).all()
        or (
            row_sums
            <= 0
        ).any()
    ):
        raise RuntimeError(
            "Invalid probability matrix."
        )

    return (
        probabilities
        / row_sums
    )

def predicted_labels(
    probabilities,
    class_order,
):
    return np.asarray(
        class_order,
        dtype=str,
    )[
        np.asarray(
            probabilities,
            dtype=float,
        ).argmax(
            axis=1
        )
    ]

def multiclass_brier(
    y_true,
    probabilities,
    class_order,
):
    y_true = np.asarray(
        y_true,
        dtype=str,
    )

    class_to_index = {
        class_label: index
        for index, class_label
        in enumerate(
            class_order
        )
    }

    indicator = np.zeros_like(
        probabilities,
        dtype=float,
    )

    for row_index, class_label in enumerate(
        y_true
    ):
        indicator[
            row_index,
            class_to_index[
                class_label
            ],
        ] = 1.0

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - indicator
                )
                ** 2,
                axis=1,
            )
        )
    )

def metric_bundle(
    y_true,
    probabilities,
    class_order,
):
    labels = predicted_labels(
        probabilities,
        class_order,
    )

    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                labels,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                labels,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                labels,
                labels=class_order,
                average="macro",
                zero_division=0,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                labels,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=class_order,
            )
        ),
        "brier_multiclass": multiclass_brier(
            y_true,
            probabilities,
            class_order,
        ),
    }

def expected_calibration_error(
    y_true,
    probabilities,
    class_order,
    n_bins,
):
    y_true = np.asarray(
        y_true,
        dtype=str,
    )

    predictions = predicted_labels(
        probabilities,
        class_order,
    )

    confidence = probabilities.max(
        axis=1
    )

    correct = (
        predictions
        == y_true
    ).astype(float)

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins
        + 1,
    )

    ece = 0.0
    maximum_gap = 0.0
    rows = []

    for bin_index in range(
        n_bins
    ):
        lower = bin_edges[
            bin_index
        ]

        upper = bin_edges[
            bin_index
            + 1
        ]

        if bin_index == (
            n_bins
            - 1
        ):
            mask = (
                confidence
                >= lower
            ) & (
                confidence
                <= upper
            )
        else:
            mask = (
                confidence
                >= lower
            ) & (
                confidence
                < upper
            )

        count = int(
            mask.sum()
        )

        if count == 0:
            continue

        empirical_accuracy = float(
            correct[
                mask
            ].mean()
        )

        mean_confidence = float(
            confidence[
                mask
            ].mean()
        )

        gap = abs(
            empirical_accuracy
            - mean_confidence
        )

        ece += (
            count
            / len(
                confidence
            )
        ) * gap

        maximum_gap = max(
            maximum_gap,
            gap,
        )

        rows.append(
            {
                "bin_index": bin_index,
                "lower": lower,
                "upper": upper,
                "count": count,
                "empirical_accuracy": empirical_accuracy,
                "mean_confidence": mean_confidence,
                "absolute_gap": gap,
            }
        )

    return (
        float(
            ece
        ),
        float(
            maximum_gap
        ),
        pd.DataFrame(
            rows
        ),
    )

def percentile_interval(
    values,
    confidence_level,
):
    alpha = (
        1.0
        - confidence_level
    ) / 2.0

    return (
        float(
            np.quantile(
                values,
                alpha,
            )
        ),
        float(
            np.quantile(
                values,
                1.0
                - alpha,
            )
        ),
    )

def stratified_group_bootstrap_indices(
    frame,
    n_bootstrap,
    seed,
):
    group_labels = (
        frame[
            [
                "duplicate_group_id",
                "class_label",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "class_label",
                "duplicate_group_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    label_counts = (
        frame.groupby(
            "duplicate_group_id"
        )[
            "class_label"
        ]
        .nunique()
    )

    if (
        label_counts
        > 1
    ).any():
        raise RuntimeError(
            "A duplicate group contains multiple labels."
        )

    group_to_indices = {
        group_id: frame.index[
            frame[
                "duplicate_group_id"
            ] == group_id
        ].to_numpy(
            dtype=int
        )
        for group_id in group_labels[
            "duplicate_group_id"
        ]
    }

    groups_by_class = {
        class_label: group_labels.loc[
            group_labels[
                "class_label"
            ] == class_label,
            "duplicate_group_id",
        ].to_numpy()
        for class_label in sorted(
            group_labels[
                "class_label"
            ].unique()
        )
    }

    rng = np.random.default_rng(
        seed
    )

    output = []

    for _ in range(
        n_bootstrap
    ):
        sampled_indices = []

        for class_groups in groups_by_class.values():
            sampled_groups = rng.choice(
                class_groups,
                size=len(
                    class_groups
                ),
                replace=True,
            )

            for group_id in sampled_groups:
                sampled_indices.extend(
                    group_to_indices[
                        group_id
                    ].tolist()
                )

        output.append(
            np.asarray(
                sampled_indices,
                dtype=int,
            )
        )

    return output

In [ ]:
# ============================================================
# 6. Exact original champion performance and reconciliation
# ============================================================
nb05_global_metrics = load_csv(
    "notebook05",
    (
        "07_results/05_BASELINE_CALIBRATION/"
        "01_baseline_test_global_metrics.csv"
    ),
)

performance_rows = []
interval_rows = []
metric_validation_rows = []

for dataset_index, dataset_id in enumerate(
    FROZEN_CHAMPIONS
):
    champion = FROZEN_CHAMPIONS[
        dataset_id
    ]

    metadata = champion_metadata_map[
        dataset_id
    ]

    class_order = metadata[
        "class_order"
    ]

    test_frame = prediction_frames[
        (
            dataset_id,
            "test",
        )
    ].reset_index(
        drop=True
    )

    y_true = test_frame[
        "class_label"
    ].to_numpy(
        dtype=str
    )

    for probability_status in [
        "native",
        "temperature_scaled",
    ]:
        probabilities = probability_matrix(
            test_frame,
            probability_status,
        )

        metrics = metric_bundle(
            y_true,
            probabilities,
            class_order,
        )

        performance_rows.append(
            {
                "dataset_id": dataset_id,
                "representation": champion[
                    "representation"
                ],
                "algorithm": champion[
                    "algorithm"
                ],
                "probability_status": probability_status,
                "n_test": len(
                    test_frame
                ),
                **metrics,
                "authoritative_source": (
                    "Notebook 05 exact frozen predictions"
                ),
            }
        )

        source_row = nb05_global_metrics.loc[
            (
                nb05_global_metrics[
                    "dataset_id"
                ] == dataset_id
            )
            & (
                nb05_global_metrics[
                    "representation"
                ] == champion[
                    "representation"
                ]
            )
            & (
                nb05_global_metrics[
                    "algorithm"
                ] == champion[
                    "algorithm"
                ]
            )
            & (
                nb05_global_metrics[
                    "probability_status"
                ] == probability_status
            )
        ].iloc[0]

        column_map = {
            "accuracy": "accuracy",
            "balanced_accuracy": (
                "balanced_accuracy"
            ),
            "macro_f1": "macro_f1",
            "mcc": "mcc",
            "log_loss": "log_loss",
            "brier_multiclass": (
                "brier_multiclass"
            ),
        }

        for metric_name, source_column in column_map.items():
            absolute_difference = abs(
                metrics[
                    metric_name
                ]
                - float(
                    source_row[
                        source_column
                    ]
                )
            )

            metric_validation_rows.append(
                {
                    "dataset_id": dataset_id,
                    "probability_status": probability_status,
                    "metric": metric_name,
                    "recomputed": metrics[
                        metric_name
                    ],
                    "notebook05_reported": float(
                        source_row[
                            source_column
                        ]
                    ),
                    "absolute_difference": absolute_difference,
                    "passed": bool(
                        absolute_difference
                        <= 1e-10
                    ),
                }
            )

    native_probabilities = probability_matrix(
        test_frame,
        "native",
    )

    native_metrics = metric_bundle(
        y_true,
        native_probabilities,
        class_order,
    )

    bootstrap_indices = (
        stratified_group_bootstrap_indices(
            test_frame,
            N_BOOTSTRAP,
            BOOTSTRAP_SEED
            + dataset_index,
        )
    )

    bootstrap_metrics = {
        metric_name: np.empty(
            N_BOOTSTRAP,
            dtype=float,
        )
        for metric_name in [
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "mcc",
            "log_loss",
            "brier_multiclass",
        ]
    }

    for bootstrap_index, sampled_indices in enumerate(
        bootstrap_indices
    ):
        metrics = metric_bundle(
            y_true[
                sampled_indices
            ],
            native_probabilities[
                sampled_indices
            ],
            class_order,
        )

        for metric_name in bootstrap_metrics:
            bootstrap_metrics[
                metric_name
            ][
                bootstrap_index
            ] = metrics[
                metric_name
            ]

    for metric_name, values in bootstrap_metrics.items():
        ci_low, ci_high = percentile_interval(
            values,
            CONFIDENCE_LEVEL,
        )

        interval_rows.append(
            {
                "dataset_id": dataset_id,
                "representation": champion[
                    "representation"
                ],
                "algorithm": champion[
                    "algorithm"
                ],
                "probability_status": "native",
                "metric": metric_name,
                "estimate": native_metrics[
                    metric_name
                ],
                "ci_low": ci_low,
                "ci_high": ci_high,
                "confidence_level": CONFIDENCE_LEVEL,
                "n_bootstrap": N_BOOTSTRAP,
                "bootstrap_unit": (
                    "duplicate_group_id"
                ),
            }
        )

primary_performance = pd.DataFrame(
    performance_rows
)

primary_performance_intervals = pd.DataFrame(
    interval_rows
)

primary_metric_reconciliation = pd.DataFrame(
    metric_validation_rows
)

if not primary_metric_reconciliation[
    "passed"
].all():
    display(
        primary_metric_reconciliation.loc[
            ~primary_metric_reconciliation[
                "passed"
            ]
        ]
    )

    raise RuntimeError(
        "Recomputed metrics do not exactly match Notebook 05."
    )

display(
    primary_performance.loc[
        primary_performance[
            "probability_status"
        ] == "native"
    ]
)

print(
    "Exact Notebook 05 metric reconciliation: PASSED"
)

In [ ]:
# ============================================================
# 7. Exact original calibration, confusion and classwise metrics
# ============================================================
calibration_rows = []
classwise_rows = []
confusion_tables = {}
classwise_validation_rows = []

nb05_class_metrics = load_csv(
    "notebook05",
    (
        "07_results/05_BASELINE_CALIBRATION/"
        "02_baseline_test_class_metrics.csv"
    ),
)

nb05_confusion_long = load_csv(
    "notebook05",
    (
        "07_results/05_BASELINE_CALIBRATION/"
        "03_baseline_test_confusion_matrices_long.csv"
    ),
)

for dataset_id in FROZEN_CHAMPIONS:
    champion = FROZEN_CHAMPIONS[
        dataset_id
    ]

    metadata = champion_metadata_map[
        dataset_id
    ]

    class_order = metadata[
        "class_order"
    ]

    for partition in [
        "calibration",
        "test",
    ]:
        frame = prediction_frames[
            (
                dataset_id,
                partition,
            )
        ]

        y_true = frame[
            "class_label"
        ].to_numpy(
            dtype=str
        )

        reliability_tables = {}

        for probability_status in [
            "native",
            "temperature_scaled",
        ]:
            probabilities = probability_matrix(
                frame,
                probability_status,
            )

            metrics = metric_bundle(
                y_true,
                probabilities,
                class_order,
            )

            ece, maximum_error, reliability = (
                expected_calibration_error(
                    y_true,
                    probabilities,
                    class_order,
                    N_RELIABILITY_BINS,
                )
            )

            reliability_tables[
                probability_status
            ] = reliability

            reliability.to_csv(
                RESULTS_DIR
                / (
                    f"reliability_bins_{dataset_id}_"
                    f"{partition}_{probability_status}.csv"
                ),
                index=False,
                encoding="utf-8-sig",
            )

            calibration_rows.append(
                {
                    "dataset_id": dataset_id,
                    "representation": champion[
                        "representation"
                    ],
                    "partition": partition,
                    "probability_status": probability_status,
                    "temperature": metadata[
                        "temperature"
                    ],
                    "log_loss": metrics[
                        "log_loss"
                    ],
                    "brier_multiclass": metrics[
                        "brier_multiclass"
                    ],
                    "ece_equal_width_15_bins": ece,
                    "maximum_calibration_error": maximum_error,
                    "authoritative_source": (
                        "Notebook 05 exact frozen predictions"
                    ),
                }
            )

        figure, axis = plt.subplots(
            figsize=(
                7,
                6,
            )
        )

        axis.plot(
            [
                0.0,
                1.0,
            ],
            [
                0.0,
                1.0,
            ],
            linestyle="--",
            linewidth=1,
            label="Perfect calibration",
        )

        for probability_status, label in [
            (
                "native",
                "Native",
            ),
            (
                "temperature_scaled",
                "Temperature-scaled",
            ),
        ]:
            reliability = reliability_tables[
                probability_status
            ]

            axis.plot(
                reliability[
                    "mean_confidence"
                ],
                reliability[
                    "empirical_accuracy"
                ],
                marker="o",
                label=label,
            )

        axis.set_xlabel(
            "Mean predicted confidence"
        )

        axis.set_ylabel(
            "Empirical accuracy"
        )

        axis.set_title(
            (
                f"Reliability diagram — "
                f"{dataset_id.replace('_', ' ').title()} — "
                f"{partition}"
            )
        )

        axis.grid(
            True,
            alpha=0.3,
        )

        axis.legend()
        figure.tight_layout()

        for extension in [
            "png",
            "pdf",
        ]:
            figure.savefig(
                FIGURES_SUPP_DIR
                / (
                    f"Figure_S_reliability_{dataset_id}_"
                    f"{partition}.{extension}"
                ),
                dpi=300
                if extension
                == "png"
                else None,
                bbox_inches="tight",
            )

        plt.show()

    test_frame = prediction_frames[
        (
            dataset_id,
            "test",
        )
    ]

    native_probabilities = probability_matrix(
        test_frame,
        "native",
    )

    y_true = test_frame[
        "class_label"
    ].to_numpy(
        dtype=str
    )

    y_pred = predicted_labels(
        native_probabilities,
        class_order,
    )

    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=class_order,
    )

    matrix_frame = pd.DataFrame(
        matrix,
        index=class_order,
        columns=class_order,
    )

    confusion_tables[
        dataset_id
    ] = matrix_frame

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=class_order,
            zero_division=0,
        )
    )

    for class_index, class_label in enumerate(
        class_order
    ):
        classwise_rows.append(
            {
                "dataset_id": dataset_id,
                "representation": champion[
                    "representation"
                ],
                "algorithm": champion[
                    "algorithm"
                ],
                "class_label": class_label,
                "precision": float(
                    precision[
                        class_index
                    ]
                ),
                "recall": float(
                    recall[
                        class_index
                    ]
                ),
                "f1": float(
                    f1[
                        class_index
                    ]
                ),
                "support": int(
                    support[
                        class_index
                    ]
                ),
                "authoritative_source": (
                    "Notebook 05 exact frozen predictions"
                ),
            }
        )

        source_row = nb05_class_metrics.loc[
            (
                nb05_class_metrics[
                    "dataset_id"
                ] == dataset_id
            )
            & (
                nb05_class_metrics[
                    "representation"
                ] == champion[
                    "representation"
                ]
            )
            & (
                nb05_class_metrics[
                    "algorithm"
                ] == champion[
                    "algorithm"
                ]
            )
            & (
                nb05_class_metrics[
                    "class_label"
                ] == class_label
            )
        ].iloc[0]

        for metric_name, recomputed_value in [
            (
                "precision",
                precision[
                    class_index
                ],
            ),
            (
                "recall",
                recall[
                    class_index
                ],
            ),
            (
                "f1",
                f1[
                    class_index
                ],
            ),
            (
                "support",
                support[
                    class_index
                ],
            ),
        ]:
            absolute_difference = abs(
                float(
                    recomputed_value
                )
                - float(
                    source_row[
                        metric_name
                    ]
                )
            )

            classwise_validation_rows.append(
                {
                    "dataset_id": dataset_id,
                    "class_label": class_label,
                    "metric": metric_name,
                    "absolute_difference": absolute_difference,
                    "passed": bool(
                        absolute_difference
                        <= 1e-10
                    ),
                }
            )

    figure, axis = plt.subplots(
        figsize=(
            8,
            7,
        )
    )

    image = axis.imshow(
        matrix,
        aspect="auto",
    )

    axis.set_xticks(
        np.arange(
            len(
                class_order
            )
        )
    )

    axis.set_yticks(
        np.arange(
            len(
                class_order
            )
        )
    )

    axis.set_xticklabels(
        class_order,
        rotation=45,
        ha="right",
    )

    axis.set_yticklabels(
        class_order,
    )

    axis.set_xlabel(
        "Predicted class"
    )

    axis.set_ylabel(
        "True class"
    )

    axis.set_title(
        (
            f"Frozen-test confusion matrix — "
            f"{dataset_id.replace('_', ' ').title()}"
        )
    )

    for row_index in range(
        matrix.shape[
            0
        ]
    ):
        for column_index in range(
            matrix.shape[
                1
            ]
        ):
            axis.text(
                column_index,
                row_index,
                str(
                    matrix[
                        row_index,
                        column_index,
                    ]
                ),
                ha="center",
                va="center",
                fontsize=8,
            )

    figure.colorbar(
        image,
        ax=axis,
        label="Count",
    )

    figure.tight_layout()

    for extension in [
        "png",
        "pdf",
    ]:
        figure.savefig(
            FIGURES_SUPP_DIR
            / (
                f"Figure_S_confusion_matrix_"
                f"{dataset_id}.{extension}"
            ),
            dpi=300
            if extension
            == "png"
            else None,
            bbox_inches="tight",
        )

    plt.show()

calibration_audit = pd.DataFrame(
    calibration_rows
)

classwise_metrics = pd.DataFrame(
    classwise_rows
)

classwise_reconciliation = pd.DataFrame(
    classwise_validation_rows
)

if not classwise_reconciliation[
    "passed"
].all():
    display(
        classwise_reconciliation.loc[
            ~classwise_reconciliation[
                "passed"
            ]
        ]
    )

    raise RuntimeError(
        "Classwise metrics do not exactly match Notebook 05."
    )

print(
    "Exact Notebook 05 classwise reconciliation: PASSED"
)

In [ ]:
# ============================================================
# 8. Original selective-classification results from Notebook 07
# ============================================================
selective_protocol = load_json(
    "notebook07",
    "03_protocol/selective_uncertainty_protocol_v1.json",
)

selective_validation = load_csv(
    "notebook07",
    (
        "07_results/07_SELECTIVE_UNCERTAINTY/"
        "05_final_selective_validation.csv"
    ),
)

selective_fixed = load_csv(
    "notebook07",
    (
        "09_tables/07_SELECTIVE_UNCERTAINTY/"
        "Table_calibration_fixed_operating_points.csv"
    ),
)

selective_intervals = load_csv(
    "notebook07",
    (
        "09_tables/07_SELECTIVE_UNCERTAINTY/"
        "Table_selective_operating_point_intervals.csv"
    ),
)

selective_classwise = load_csv(
    "notebook07",
    (
        "09_tables/07_SELECTIVE_UNCERTAINTY/"
        "Table_primary_classwise_90pct_coverage.csv"
    ),
)

risk_coverage_curves = load_csv(
    "notebook07",
    (
        "07_results/07_SELECTIVE_UNCERTAINTY/"
        "03_test_risk_coverage_curves.csv"
    ),
)

selective_primary = selective_fixed.loc[
    (
        selective_fixed[
            "probability_status"
        ] == "temperature_scaled"
    )
    & (
        as_boolean_series(
            selective_fixed[
                "selected_in_calibration"
            ]
        )
    )
    & (
        np.isclose(
            selective_fixed[
                "target_calibration_coverage"
            ],
            0.90,
        )
    )
].copy()

if len(
    selective_primary
) != 2:
    raise RuntimeError(
        "Expected exactly two primary selective rows."
    )

for row in selective_primary.itertuples():
    expected = FROZEN_CHAMPIONS[
        row.dataset_id
    ]

    if (
        row.representation
        != expected[
            "representation"
        ]
        or row.algorithm
        != expected[
            "algorithm"
        ]
    ):
        raise RuntimeError(
            "Notebook 07 primary model does not match "
            "the frozen champion."
        )

figure, axis = plt.subplots(
    figsize=(
        8,
        6,
    )
)

for dataset_id, champion in FROZEN_CHAMPIONS.items():
    selected_score = selective_primary.loc[
        selective_primary[
            "dataset_id"
        ] == dataset_id,
        "certainty_score",
    ].iloc[0]

    subset = risk_coverage_curves.loc[
        (
            risk_coverage_curves[
                "dataset_id"
            ] == dataset_id
        )
        & (
            risk_coverage_curves[
                "probability_status"
            ] == "temperature_scaled"
        )
        & (
            risk_coverage_curves[
                "certainty_score"
            ] == selected_score
        )
    ].sort_values(
        "coverage"
    )

    step = max(
        1,
        len(
            subset
        )
        // 200
    )

    subset = subset.iloc[
        ::step
    ]

    axis.plot(
        subset[
            "coverage"
        ],
        subset[
            "selective_risk"
        ],
        label=dataset_id.replace(
            "_",
            " ",
        ).title(),
    )

axis.set_xlabel(
    "Empirical test coverage"
)

axis.set_ylabel(
    "Selective risk"
)

axis.set_title(
    "Risk–coverage curves of the exact frozen champions"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()
figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_4_risk_coverage_curves.{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

display(selective_primary)

In [ ]:
# ============================================================
# 9. Original canonical conformal results from Notebook 08 v2
# ============================================================
conformal_protocol = load_json(
    "notebook08",
    (
        "03_protocol/"
        "canonical_conformal_prediction_protocol_v2.json"
    ),
)

conformal_validation = load_csv(
    "notebook08",
    (
        "07_results/08_V2_CANONICAL_CONFORMAL/"
        "05_final_conformal_validation_v2.csv"
    ),
)

conformal_primary = load_csv(
    "notebook08",
    (
        "07_results/08_V2_CANONICAL_CONFORMAL/"
        "03_primary_conformal_results_v2.csv"
    ),
)

conformal_primary_classwise = load_csv(
    "notebook08",
    (
        "07_results/08_V2_CANONICAL_CONFORMAL/"
        "04_primary_conformal_classwise_v2.csv"
    ),
)

conformal_global_all = load_csv(
    "notebook08",
    (
        "09_tables/08_V2_CANONICAL_CONFORMAL/"
        "Table_conformal_global_test_results_v2.csv"
    ),
)

conformal_classwise_all = load_csv(
    "notebook08",
    (
        "09_tables/08_V2_CANONICAL_CONFORMAL/"
        "Table_conformal_classwise_test_results_v2.csv"
    ),
)

conformal_quantiles = load_csv(
    "notebook08",
    (
        "09_tables/08_V2_CANONICAL_CONFORMAL/"
        "Table_conformal_calibration_quantiles_v2.csv"
    ),
)

conformal_global_intervals = load_csv(
    "notebook08",
    (
        "09_tables/08_V2_CANONICAL_CONFORMAL/"
        "Table_conformal_global_intervals_v2.csv"
    ),
)

conformal_paired = load_csv(
    "notebook08",
    (
        "09_tables/08_V2_CANONICAL_CONFORMAL/"
        "Table_conformal_paired_comparisons_v2.csv"
    ),
)

for row in conformal_primary.itertuples():
    expected = FROZEN_CHAMPIONS[
        row.dataset_id
    ]

    if (
        row.representation
        != expected[
            "representation"
        ]
        or row.algorithm
        != expected[
            "algorithm"
        ]
    ):
        raise RuntimeError(
            "Notebook 08 primary model does not match "
            "the frozen champion."
        )

conformal_primary = conformal_primary.copy()

conformal_primary[
    "nonempty_coverage_upper_bound"
] = (
    1.0
    - conformal_primary[
        "empty_set_rate"
    ]
)

conformal_primary[
    "margin_below_nonempty_upper_bound"
] = (
    conformal_primary[
        "nonempty_coverage_upper_bound"
    ]
    - conformal_primary[
        "coverage"
    ]
)

figure, axis = plt.subplots(
    figsize=(
        8,
        6,
    )
)

display_labels = [
    (
        f"{row.dataset_id}\n"
        f"{int(row.nominal_coverage * 100)}%"
    )
    for row in conformal_primary.itertuples()
]

positions = np.arange(
    len(
        conformal_primary
    )
)

axis.bar(
    positions,
    conformal_primary[
        "coverage"
    ],
)

axis.scatter(
    positions,
    conformal_primary[
        "nominal_coverage"
    ],
    marker="x",
    label="Nominal coverage",
)

axis.set_xticks(
    positions
)

axis.set_xticklabels(
    display_labels,
    rotation=30,
    ha="right",
)

axis.set_ylabel(
    "Coverage"
)

axis.set_title(
    "Canonical randomized APS coverage"
)

axis.grid(
    True,
    axis="y",
    alpha=0.3,
)

axis.legend()
figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_5_conformal_APS_summary.{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

display(conformal_primary)

In [ ]:
# ============================================================
# 10. Confirmatory reviewer reanalysis from Notebook 12
# ============================================================
NB12_TABLE_ROOT = (
    source_roots[
        "notebook12"
    ]
    / "09_tables"
    / "12_REVIEWER_REANALYSIS"
)

def nb12_table(
    filename,
):
    path = (
        NB12_TABLE_ROOT
        / filename
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    return pd.read_csv(
        path
    )

representation_audit = nb12_table(
    "Table_01_representation_audit.csv"
)

representation_metrics = nb12_table(
    "Table_05_corrected_representation_test_metrics.csv"
)

paired_contrasts = nb12_table(
    "Table_06_corrected_paired_representation_contrasts.csv"
)

mcnemar_results = nb12_table(
    "Table_07_exact_mcnemar_discordant_counts.csv"
)

rice_equivalence = nb12_table(
    "Table_08_rice_bootstrap_TOST_equivalence.csv"
)

controlled_importance = nb12_table(
    "Table_09_controlled_R1_size_shape_permutation.csv"
)

controlled_scale = nb12_table(
    "Table_10_controlled_R1_scale_bias.csv"
)

naive_leakage = nb12_table(
    "Table_17_repeated_naive_row_split_leakage_audit.csv"
)

fixed_leakage = nb12_table(
    "Table_18_fixed_test_duplicate_contamination_audit.csv"
)

nb12_timing = nb12_table(
    "Table_04_computational_timing.csv"
)

dry_complete_features = int(
    representation_audit.loc[
        (
            representation_audit[
                "dataset_id"
            ] == "dry_bean"
        )
        & (
            representation_audit[
                "representation"
            ] == "R3_complete_invariant"
        ),
        "n_features",
    ].iloc[0]
)

if dry_complete_features != 8:
    raise RuntimeError(
        "The corrected Dry Bean invariant representation "
        "does not contain eight features."
    )

original_rice_features = set(
    champion_metadata_map[
        "iniap_rice"
    ][
        "feature_columns"
    ]
)

reviewer_rice_features = set(
    value.strip()
    for value in representation_audit.loc[
        (
            representation_audit[
                "dataset_id"
            ] == "iniap_rice"
        )
        & (
            representation_audit[
                "representation"
            ] == "R4_complete_log_area"
        ),
        "features",
    ].iloc[0].split(
        "|"
    )
)

if (
    original_rice_features
    != reviewer_rice_features
):
    raise RuntimeError(
        "The original Rice R4 feature set and the Notebook 12 "
        "R4-complete feature set are not identical."
    )

representations_to_plot = [
    "R1_full_native",
    "R2_measured_core",
    "R3_complete_invariant",
    "R4_complete_log_area",
]

representation_plot = representation_metrics.loc[
    representation_metrics[
        "representation"
    ].isin(
        representations_to_plot
    )
].copy()

representation_plot[
    "display_label"
] = (
    representation_plot[
        "dataset_id"
    ]
    + "\n"
    + representation_plot[
        "representation"
    ]
)

figure, axis = plt.subplots(
    figsize=(
        11,
        6,
    )
)

positions = np.arange(
    len(
        representation_plot
    )
)

axis.bar(
    positions,
    representation_plot[
        "macro_f1"
    ],
)

axis.set_xticks(
    positions
)

axis.set_xticklabels(
    representation_plot[
        "display_label"
    ],
    rotation=45,
    ha="right",
)

axis.set_ylabel(
    "Frozen-test macro-F1"
)

axis.set_title(
    "Corrected confirmatory representation comparison"
)

axis.set_ylim(
    0.70,
    1.00,
)

axis.grid(
    True,
    axis="y",
    alpha=0.3,
)

figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_1_corrected_representation_performance."
            f"{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

key_contrasts = paired_contrasts.loc[
    (
        paired_contrasts[
            "metric"
        ] == "macro_f1"
    )
    & (
        paired_contrasts[
            "representation_a"
        ] == "R1_full_native"
    )
    & (
        paired_contrasts[
            "representation_b"
        ].isin(
            [
                "R2_measured_core",
                "R3_complete_invariant",
                "R4_complete_log_area",
            ]
        )
    )
].copy()

key_contrasts[
    "display_label"
] = (
    key_contrasts[
        "dataset_id"
    ]
    + ": "
    + key_contrasts[
        "representation_a"
    ]
    + " − "
    + key_contrasts[
        "representation_b"
    ]
)

figure, axis = plt.subplots(
    figsize=(
        10,
        6,
    )
)

positions = np.arange(
    len(
        key_contrasts
    )
)

axis.errorbar(
    key_contrasts[
        "delta_a_minus_b"
    ],
    positions,
    xerr=np.vstack(
        [
            key_contrasts[
                "delta_a_minus_b"
            ]
            - key_contrasts[
                "ci_low"
            ],
            key_contrasts[
                "ci_high"
            ]
            - key_contrasts[
                "delta_a_minus_b"
            ],
        ]
    ),
    fmt="o",
    capsize=4,
)

axis.axvline(
    0.0,
    linestyle="--",
    linewidth=1,
)

axis.set_yticks(
    positions
)

axis.set_yticklabels(
    key_contrasts[
        "display_label"
    ]
)

axis.set_xlabel(
    "Paired macro-F1 difference"
)

axis.set_title(
    "Corrected paired representation contrasts"
)

axis.grid(
    True,
    axis="x",
    alpha=0.3,
)

figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_2_corrected_paired_contrasts."
            f"{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

importance_plot = controlled_importance.copy()

importance_plot[
    "display_label"
] = (
    importance_plot[
        "dataset_id"
    ]
    + "\n"
    + importance_plot[
        "analysis"
    ]
    + "\n"
    + importance_plot[
        "block"
    ]
)

figure, axis = plt.subplots(
    figsize=(
        12,
        6,
    )
)

positions = np.arange(
    len(
        importance_plot
    )
)

axis.bar(
    positions,
    importance_plot[
        "importance_mean"
    ],
    yerr=importance_plot[
        "importance_sd"
    ],
    capsize=4,
)

axis.set_xticks(
    positions
)

axis.set_xticklabels(
    importance_plot[
        "display_label"
    ],
    rotation=45,
    ha="right",
)

axis.set_ylabel(
    "Macro-F1 decrease"
)

axis.set_title(
    "Controlled R1 block importance"
)

axis.grid(
    True,
    axis="y",
    alpha=0.3,
)

figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_3_controlled_R1_block_importance."
            f"{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

figure, axis = plt.subplots(
    figsize=(
        8,
        6,
    )
)

for dataset_id in sorted(
    controlled_scale[
        "dataset_id"
    ].unique()
):
    subset = controlled_scale.loc[
        controlled_scale[
            "dataset_id"
        ] == dataset_id
    ].sort_values(
        "scale_bias_percent"
    )

    axis.plot(
        subset[
            "scale_bias_percent"
        ],
        subset[
            "macro_f1_drop"
        ],
        marker="o",
        label=dataset_id.replace(
            "_",
            " ",
        ).title(),
    )

axis.axhline(
    0.02,
    linestyle="--",
    linewidth=1,
    label="Prespecified practical-loss threshold",
)

axis.set_xlabel(
    "Coherent image-space scale bias (%)"
)

axis.set_ylabel(
    "Macro-F1 decrease"
)

axis.set_title(
    "Controlled R1 image-space scale sensitivity"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()
figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_6_controlled_R1_scale_sensitivity."
            f"{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

In [ ]:
# ============================================================
# 11. Manuscript-ready and supplementary table export
# ============================================================
def save_table(
    frame,
    directory,
    stem,
    float_format="%.6f",
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    frame.to_csv(
        directory
        / f"{stem}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    (
        directory
        / f"{stem}.md"
    ).write_text(
        frame.to_markdown(
            index=False,
            floatfmt=".6f",
        ),
        encoding="utf-8",
    )

    (
        directory
        / f"{stem}.tex"
    ).write_text(
        frame.to_latex(
            index=False,
            float_format=lambda value: (
                float_format
                % value
            ),
            escape=True,
        ),
        encoding="utf-8",
    )

# Main Table 1.
main_table_1 = frozen_model_catalog[
    [
        "dataset_id",
        "representation",
        "algorithm",
        "candidate_id",
        "temperature",
        "train_records",
        "calibration_records",
        "test_records",
        "feature_count",
        "source",
    ]
].copy()

# Main Table 2.
main_table_2 = primary_performance.loc[
    primary_performance[
        "probability_status"
    ] == "native"
][
    [
        "dataset_id",
        "representation",
        "algorithm",
        "n_test",
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
        "mcc",
        "log_loss",
        "brier_multiclass",
        "authoritative_source",
    ]
].copy()

# Main Table 3.
main_table_3 = key_contrasts[
    [
        "dataset_id",
        "representation_a",
        "representation_b",
        "delta_a_minus_b",
        "ci_low",
        "ci_high",
        "bootstrap_p_raw",
        "bootstrap_p_holm",
        "reject_holm_0_05",
    ]
].copy()

# Main Table 4.
main_table_4 = selective_primary[
    [
        "dataset_id",
        "representation",
        "algorithm",
        "probability_status",
        "certainty_score",
        "target_calibration_coverage",
        "certainty_threshold",
        "test_coverage",
        "test_selective_accuracy",
        "test_selective_risk",
        "test_macro_f1",
        "test_min_class_coverage",
    ]
].copy()

# Main Table 5.
main_table_5 = conformal_primary[
    [
        "dataset_id",
        "representation",
        "algorithm",
        "probability_status",
        "method",
        "nominal_coverage",
        "coverage",
        "coverage_gap",
        "mean_set_size",
        "singleton_rate",
        "empty_set_rate",
        "nonempty_coverage_upper_bound",
        "margin_below_nonempty_upper_bound",
    ]
].copy()

# Main Table 6.
main_table_6 = controlled_importance[
    [
        "dataset_id",
        "representation",
        "analysis",
        "block",
        "n_block_features",
        "matched_subset_size",
        "importance_mean",
        "importance_sd",
        "importance_per_feature",
    ]
].copy()

# Main Table 7.
main_table_7 = controlled_scale.loc[
    controlled_scale[
        "scale_bias_percent"
    ].isin(
        [
            -5.0,
            -2.0,
            0.0,
            2.0,
            5.0,
        ]
    )
][
    [
        "dataset_id",
        "representation",
        "scale_bias_percent",
        "macro_f1",
        "macro_f1_drop",
        "accuracy",
        "mcc",
        "label_flip_rate",
        "interpretation",
    ]
].copy()

main_tables = {
    "Table_1_frozen_models_and_provenance": main_table_1,
    "Table_2_exact_frozen_test_performance": main_table_2,
    "Table_3_corrected_representation_contrasts": main_table_3,
    "Table_4_original_selective_classification": main_table_4,
    "Table_5_original_canonical_APS": main_table_5,
    "Table_6_controlled_R1_block_importance": main_table_6,
    "Table_7_controlled_R1_scale_sensitivity": main_table_7,
}

source_reconciliation = pd.DataFrame(
    [
        {
            "analysis_layer": (
                "Primary performance, calibration, confusion "
                "and classwise metrics"
            ),
            "authoritative_source": "Notebook 05",
            "models": (
                "Dry Bean R1 + RBF-SVM; "
                "Rice R4 hybrid log-area + RBF-SVM"
            ),
            "refitted_notebook12_predictions_used": False,
        },
        {
            "analysis_layer": "Selective classification",
            "authoritative_source": "Notebook 07",
            "models": "Exact frozen Notebook 05 champions",
            "refitted_notebook12_predictions_used": False,
        },
        {
            "analysis_layer": "Canonical conformal prediction",
            "authoritative_source": "Notebook 08 v2",
            "models": "Exact frozen Notebook 05 champions",
            "refitted_notebook12_predictions_used": False,
        },
        {
            "analysis_layer": (
                "Corrected representations, exact McNemar, "
                "equivalence, controlled importance, scale and leakage"
            ),
            "authoritative_source": "Notebook 12 v2",
            "models": (
                "Confirmatory reviewer-driven reanalysis"
            ),
            "refitted_notebook12_predictions_used": True,
        },
    ]
)

supplementary_tables = {
    "Table_S1_source_reconciliation": (
        source_reconciliation
    ),
    "Table_S2_exact_model_metadata": (
        frozen_model_catalog
    ),
    "Table_S3_primary_metric_bootstrap_intervals": (
        primary_performance_intervals
    ),
    "Table_S4_primary_metric_reconciliation": (
        primary_metric_reconciliation
    ),
    "Table_S5_calibration_audit": (
        calibration_audit
    ),
    "Table_S6_classwise_metrics": (
        classwise_metrics
    ),
    "Table_S7_complete_representation_audit": (
        representation_audit
    ),
    "Table_S8_all_representation_metrics": (
        representation_metrics
    ),
    "Table_S9_all_paired_representation_contrasts": (
        paired_contrasts
    ),
    "Table_S10_exact_McNemar_results": (
        mcnemar_results
    ),
    "Table_S11_rice_equivalence_results": (
        rice_equivalence
    ),
    "Table_S12_all_selective_operating_points": (
        selective_fixed
    ),
    "Table_S13_selective_operating_point_intervals": (
        selective_intervals
    ),
    "Table_S14_selective_classwise_90pct": (
        selective_classwise
    ),
    "Table_S15_all_conformal_global_results": (
        conformal_global_all
    ),
    "Table_S16_all_conformal_classwise_results": (
        conformal_classwise_all
    ),
    "Table_S17_conformal_quantiles": (
        conformal_quantiles
    ),
    "Table_S18_conformal_global_intervals": (
        conformal_global_intervals
    ),
    "Table_S19_conformal_paired_comparisons": (
        conformal_paired
    ),
    "Table_S20_controlled_R1_importance": (
        controlled_importance
    ),
    "Table_S21_complete_R1_scale_sensitivity": (
        controlled_scale
    ),
    "Table_S22_naive_row_split_audit": (
        naive_leakage
    ),
    "Table_S23_fixed_test_duplicate_contamination": (
        fixed_leakage
    ),
    "Table_S24_notebook12_computational_timing": (
        nb12_timing
    ),
    "Table_S25_source_validation_summary": (
        source_validation
    ),
}

for table_name, table_frame in main_tables.items():
    save_table(
        table_frame,
        TABLES_MAIN_DIR,
        table_name,
    )

for table_name, table_frame in supplementary_tables.items():
    save_table(
        table_frame,
        TABLES_SUPP_DIR,
        table_name,
    )

for dataset_id, matrix_frame in confusion_tables.items():
    save_table(
        matrix_frame.reset_index().rename(
            columns={
                "index": "true_class"
            }
        ),
        TABLES_SUPP_DIR,
        (
            f"Table_S_confusion_matrix_{dataset_id}"
        ),
        float_format="%.0f",
    )

figure_caption_catalog = pd.DataFrame(
    [
        {
            "figure": "Figure 1",
            "caption_note": (
                "Confirmatory representation comparison from "
                "Notebook 12; primary performance remains sourced "
                "from the exact Notebook 05 champions."
            ),
        },
        {
            "figure": "Figure 2",
            "caption_note": (
                "Intervals are paired duplicate-group bootstrap "
                "intervals."
            ),
        },
        {
            "figure": "Figure 3",
            "caption_note": (
                "Error bars are standard deviations across permutation "
                "repeats, not sampling confidence intervals."
            ),
        },
        {
            "figure": "Figure 4",
            "caption_note": (
                "Risk–coverage curves are from Notebook 07 and use "
                "the exact frozen champions."
            ),
        },
        {
            "figure": "Figure 5",
            "caption_note": (
                "Empty conformal sets are retained and counted as "
                "non-covering outcomes."
            ),
        },
        {
            "figure": "Figure 6",
            "caption_note": (
                "Scale perturbations are coherent transformations "
                "of image-space measurements and are not calibrated "
                "physical-size changes."
            ),
        },
    ]
)

figure_caption_catalog.to_csv(
    RESULTS_DIR
    / "figure_caption_catalog.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Main tables exported:",
    len(
        main_tables
    ),
)

print(
    "Supplementary tables exported:",
    len(
        supplementary_tables
    )
    + len(
        confusion_tables
    ),
)

In [ ]:
# ============================================================
# 12. Numerical claims and reviewer-response evidence map
# ============================================================
def primary_metric_value(
    dataset_id,
    metric,
):
    return float(
        primary_performance.loc[
            (
                primary_performance[
                    "dataset_id"
                ] == dataset_id
            )
            & (
                primary_performance[
                    "probability_status"
                ] == "native"
            ),
            metric,
        ].iloc[0]
    )

def macro_contrast(
    dataset_id,
    representation_a,
    representation_b,
):
    return paired_contrasts.loc[
        (
            paired_contrasts[
                "dataset_id"
            ] == dataset_id
        )
        & (
            paired_contrasts[
                "representation_a"
            ] == representation_a
        )
        & (
            paired_contrasts[
                "representation_b"
            ] == representation_b
        )
        & (
            paired_contrasts[
                "metric"
            ] == "macro_f1"
        )
    ].iloc[0]

dry_r1_r3 = macro_contrast(
    "dry_bean",
    "R1_full_native",
    "R3_complete_invariant",
)

dry_r1_r4 = macro_contrast(
    "dry_bean",
    "R1_full_native",
    "R4_complete_log_area",
)

claim_register = pd.DataFrame(
    [
        {
            "claim_id": "C001",
            "claim": (
                "Dry Bean frozen-test accuracy = "
                f"{primary_metric_value('dry_bean', 'accuracy'):.6f}."
            ),
            "source": "Notebook 05 / Main Table 2",
        },
        {
            "claim_id": "C002",
            "claim": (
                "Dry Bean frozen-test macro-F1 = "
                f"{primary_metric_value('dry_bean', 'macro_f1'):.6f}."
            ),
            "source": "Notebook 05 / Main Table 2",
        },
        {
            "claim_id": "C003",
            "claim": (
                "Dry Bean frozen-test MCC = "
                f"{primary_metric_value('dry_bean', 'mcc'):.6f}."
            ),
            "source": "Notebook 05 / Main Table 2",
        },
        {
            "claim_id": "C004",
            "claim": (
                "INIAP Rice frozen-test accuracy = "
                f"{primary_metric_value('iniap_rice', 'accuracy'):.6f}."
            ),
            "source": "Notebook 05 / Main Table 2",
        },
        {
            "claim_id": "C005",
            "claim": (
                "INIAP Rice frozen-test macro-F1 = "
                f"{primary_metric_value('iniap_rice', 'macro_f1'):.6f}."
            ),
            "source": "Notebook 05 / Main Table 2",
        },
        {
            "claim_id": "C006",
            "claim": (
                "The corrected Dry Bean R1–R3-complete macro-F1 "
                f"difference = {dry_r1_r3['delta_a_minus_b']:.6f}, "
                f"95% CI {dry_r1_r3['ci_low']:.6f} to "
                f"{dry_r1_r3['ci_high']:.6f}."
            ),
            "source": "Notebook 12 / Main Table 3",
        },
        {
            "claim_id": "C007",
            "claim": (
                "The corrected Dry Bean R1–R4-complete macro-F1 "
                f"difference = {dry_r1_r4['delta_a_minus_b']:.6f}, "
                f"95% CI {dry_r1_r4['ci_low']:.6f} to "
                f"{dry_r1_r4['ci_high']:.6f}."
            ),
            "source": "Notebook 12 / Main Table 3",
        },
        {
            "claim_id": "C008",
            "claim": (
                "Selective classification uses the exact frozen "
                "Notebook 05 champions and calibration-only thresholds."
            ),
            "source": "Notebook 07 / Main Table 4",
        },
        {
            "claim_id": "C009",
            "claim": (
                "Canonical APS uses the exact frozen Notebook 05 "
                "champions and calibration-only finite-sample quantiles."
            ),
            "source": "Notebook 08 v2 / Main Table 5",
        },
        {
            "claim_id": "C010",
            "claim": (
                "Empty conformal sets are retained and counted as "
                "non-covering outcomes."
            ),
            "source": "Notebook 08 v2 protocol / Main Table 5",
        },
        {
            "claim_id": "C011",
            "claim": (
                "The complete Dry Bean invariant representation "
                "contains eight dimensionless predictors."
            ),
            "source": "Notebook 12 / Table S7",
        },
        {
            "claim_id": "C012",
            "claim": (
                "The original Rice R4-hybrid feature set is identical "
                "to the Notebook 12 R4-complete feature set."
            ),
            "source": "Notebook 05 metadata and Notebook 12 audit",
        },
        {
            "claim_id": "C013",
            "claim": (
                "Linear and area-related descriptors are interpreted "
                "in pixels and squared pixels, respectively."
            ),
            "source": "Final reconciliation protocol",
        },
        {
            "claim_id": "C014",
            "claim": (
                "Notebook 12 refitted probabilities are not used for "
                "the primary performance, selective or conformal tables."
            ),
            "source": "Table S1 source reconciliation",
        },
    ]
)

claim_register.to_csv(
    CLAIMS_DIR
    / "numerical_claim_register.csv",
    index=False,
    encoding="utf-8-sig",
)

reviewer_response_map = pd.DataFrame(
    [
        {
            "reviewer_item": "M1",
            "status": "Resolved",
            "evidence": (
                "Tables 3, S7–S10; Figures 1–2"
            ),
            "response_action": (
                "Define R3-complete and retain the old reduced R3 "
                "only as a sensitivity representation."
            ),
        },
        {
            "reviewer_item": "M2",
            "status": "Resolved",
            "evidence": (
                "Tables 6–7, S20–S21; Figures 3 and 6"
            ),
            "response_action": (
                "Separate model reliance from representation sufficiency."
            ),
        },
        {
            "reviewer_item": "M3",
            "status": "Resolved",
            "evidence": "Tables 3, S9–S10",
            "response_action": (
                "Report metric bootstrap and exact McNemar separately."
            ),
        },
        {
            "reviewer_item": "M4",
            "status": "Manuscript action",
            "evidence": "",
            "response_action": (
                "Add a verified numerical comparison with published "
                "Dry Bean benchmark studies."
            ),
        },
        {
            "reviewer_item": "M5",
            "status": "Clarified",
            "evidence": (
                "Table 1 and final reconciliation protocol"
            ),
            "response_action": (
                "State that measurements are image-space pixel descriptors "
                "and provide the real repository URL and license."
            ),
        },
        {
            "reviewer_item": "M6",
            "status": "Resolved",
            "evidence": "Tables 3, S8–S9; Figures 1–2",
            "response_action": "Report and discuss R2.",
        },
        {
            "reviewer_item": "M7",
            "status": "Resolved",
            "evidence": "Table S11",
            "response_action": (
                "Use equivalence wording only where supported."
            ),
        },
        {
            "reviewer_item": "M8",
            "status": "Resolved",
            "evidence": (
                "Table S5 and reliability diagrams"
            ),
            "response_action": (
                "Describe temperature scaling as marginal or neutral."
            ),
        },
        {
            "reviewer_item": "M9",
            "status": "Resolved",
            "evidence": "Tables 5, S15–S19; Figure 5",
            "response_action": (
                "Report empty-set rates and nonempty coverage bounds."
            ),
        },
        {
            "reviewer_item": "M10",
            "status": "Resolved",
            "evidence": "Tables S22–S23",
            "response_action": (
                "Quantify leakage sensitivity and describe grouping "
                "as a validity safeguard."
            ),
        },
        {
            "reviewer_item": "M11",
            "status": "Resolved",
            "evidence": "Tables 4, S12–S13",
            "response_action": (
                "State row- or group-level evaluation explicitly."
            ),
        },
        {
            "reviewer_item": "M12",
            "status": "Resolved",
            "evidence": (
                "Table S6, confusion tables and figures"
            ),
            "response_action": (
                "Report classwise metrics and confusion matrices."
            ),
        },
        {
            "reviewer_item": "M13",
            "status": "Manuscript action",
            "evidence": "",
            "response_action": (
                "Integrate verified 2023–2026 literature."
            ),
        },
        {
            "reviewer_item": "M14",
            "status": "Repository action",
            "evidence": (
                "Final ZIP, claim register and SHA-256 manifest"
            ),
            "response_action": (
                "Deposit permitted data and code, then provide DOI "
                "and license."
            ),
        },
    ]
)

reviewer_response_map.to_csv(
    RESULTS_DIR
    / "reviewer_response_evidence_map.csv",
    index=False,
    encoding="utf-8-sig",
)

display(claim_register)
display(reviewer_response_map)

In [ ]:
# ============================================================
# 13. Final protocol, validation and integrity manifest
# ============================================================
final_protocol = {
    "protocol_name": (
        "Grain_AI_MDPI_final_reconciliation_protocol_v1"
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_zip_sha256": {
        source_key: sha256_file(
            source_path
        )
        for source_key, source_path
        in source_paths.items()
    },
    "frozen_primary_models": FROZEN_CHAMPIONS,
    "authoritative_source_policy": {
        "primary_performance_calibration_confusion": (
            "Notebook 05 exact frozen predictions"
        ),
        "selective_classification": (
            "Notebook 07 outputs based on the exact frozen champions"
        ),
        "conformal_prediction": (
            "Notebook 08 v2 outputs based on the exact frozen champions"
        ),
        "reviewer_confirmatory_reanalysis": (
            "Notebook 12 v2"
        ),
    },
    "notebook12_refitted_predictions_used_for_primary_tables": False,
    "bootstrap": {
        "repetitions": N_BOOTSTRAP,
        "unit": "duplicate_group_id",
        "stratification": "class_label",
        "confidence_level": CONFIDENCE_LEVEL,
        "seed": BOOTSTRAP_SEED,
    },
    "measurement_units": {
        "linear_descriptors": "pixels",
        "area_descriptors": "squared pixels",
        "ratios_and_shape_factors": "dimensionless",
        "pixel_to_millimetre_conversion": None,
    },
    "restrictions": [
        "No model training.",
        "No model or representation reselection.",
        "No test-based threshold selection.",
        "No test-based conformal quantile selection.",
        "No physical-unit interpretation.",
        "No fabricated benchmark result, reference, repository DOI or license.",
        "Permutation SD is not a sampling confidence interval.",
        "Empty conformal sets remain non-covering outcomes.",
    ],
}

(
    PROTOCOL_DIR
    / "final_reconciliation_protocol.json"
).write_text(
    json.dumps(
        final_protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

required_main_tables = {
    path.stem
    for path in TABLES_MAIN_DIR.glob(
        "*.csv"
    )
}

expected_main_tables = set(
    main_tables.keys()
)

required_main_figures = [
    "Figure_1_corrected_representation_performance.png",
    "Figure_2_corrected_paired_contrasts.png",
    "Figure_3_controlled_R1_block_importance.png",
    "Figure_4_risk_coverage_curves.png",
    "Figure_5_conformal_APS_summary.png",
    "Figure_6_controlled_R1_scale_sensitivity.png",
]

validation_checks = {
    "all_four_source_validations_passed": bool(
        source_validation[
            "all_checks_passed"
        ].all()
    ),
    "all_source_manifest_entries_validated": bool(
        source_manifest_validation[
            "passed"
        ].all()
    ),
    "exact_nb05_primary_metrics_reconciled": bool(
        primary_metric_reconciliation[
            "passed"
        ].all()
    ),
    "exact_nb05_classwise_metrics_reconciled": bool(
        classwise_reconciliation[
            "passed"
        ].all()
    ),
    "dry_bean_primary_is_original_R1": (
        FROZEN_CHAMPIONS[
            "dry_bean"
        ][
            "representation"
        ]
        == "R1_full_native"
    ),
    "rice_primary_is_original_R4_hybrid": (
        FROZEN_CHAMPIONS[
            "iniap_rice"
        ][
            "representation"
        ]
        == "R4_hybrid_log_area"
    ),
    "selective_models_match_exact_champions": (
        len(
            selective_primary
        )
        == 2
    ),
    "conformal_models_match_exact_champions": (
        len(
            conformal_primary
        )
        == 4
    ),
    "dry_complete_invariant_has_eight_features": (
        dry_complete_features
        == 8
    ),
    "rice_R4_original_and_reanalysis_features_identical": (
        original_rice_features
        == reviewer_rice_features
    ),
    "seven_main_tables_created": (
        required_main_tables
        == expected_main_tables
    ),
    "six_main_figures_created": all(
        (
            FIGURES_MAIN_DIR
            / filename
        ).exists()
        for filename in required_main_figures
    ),
    "reviewer_map_contains_M1_to_M14": (
        set(
            reviewer_response_map[
                "reviewer_item"
            ]
        )
        == {
            f"M{index}"
            for index in range(
                1,
                15,
            )
        }
    ),
    "claim_register_created": (
        len(
            claim_register
        )
        >= 14
    ),
    "notebook12_refit_not_used_for_primary_tables": (
        not final_protocol[
            "notebook12_refitted_predictions_used_for_primary_tables"
        ]
    ),
    "measurement_domain_is_image_space": (
        final_protocol[
            "measurement_units"
        ][
            "pixel_to_millimetre_conversion"
        ]
        is None
    ),
    "no_training_or_reselection": True,
}

final_validation = pd.DataFrame(
    [
        {
            "check": check,
            "passed": bool(
                passed
            ),
        }
        for check, passed
        in validation_checks.items()
    ]
)

final_validation.to_csv(
    RESULTS_DIR
    / "final_notebook13B_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

if not final_validation[
    "passed"
].all():
    display(
        final_validation.loc[
            ~final_validation[
                "passed"
            ]
        ]
    )

    raise RuntimeError(
        "At least one Notebook 13B validation failed."
    )

manifest_rows = []

for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if path.is_file():
        manifest_rows.append(
            {
                "relative_path": str(
                    path.relative_to(
                        OUTPUT_ROOT
                    )
                ),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(
                    path
                ),
            }
        )

output_manifest = pd.DataFrame(
    manifest_rows
)

output_manifest.to_csv(
    RESULTS_DIR
    / "notebook13B_output_integrity_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Notebook 13B final validation: PASSED"
)

display(final_validation)

print(
    "Files registered before manifest creation:",
    len(
        output_manifest
    ),
)

In [ ]:
# ============================================================
# 14. Export reconciled assets to Google Drive and ZIP
# ============================================================
if DRIVE_PROJECT_ROOT is not None:
    DRIVE_OUTPUT_DIR = (
        DRIVE_PROJECT_ROOT
        / "NOTEBOOK13B_RECONCILED_FINAL_MANUSCRIPT_ASSETS"
    )

    DRIVE_OUTPUT_ZIP = (
        DRIVE_PROJECT_ROOT
        / "NOTEBOOK13B_RECONCILED_FINAL_MANUSCRIPT_ASSETS.zip"
    )

    if DRIVE_OUTPUT_DIR.exists():
        shutil.rmtree(
            DRIVE_OUTPUT_DIR
        )

    shutil.copytree(
        OUTPUT_ROOT,
        DRIVE_OUTPUT_DIR,
    )

    if DRIVE_OUTPUT_ZIP.exists():
        DRIVE_OUTPUT_ZIP.unlink()

    with zipfile.ZipFile(
        DRIVE_OUTPUT_ZIP,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as archive:
        for path in sorted(
            OUTPUT_ROOT.rglob("*")
        ):
            if path.is_file():
                archive.write(
                    path,
                    arcname=str(
                        path.relative_to(
                            OUTPUT_ROOT
                        )
                    ),
                )

    print(
        "Drive output directory:",
        DRIVE_OUTPUT_DIR,
    )

    print(
        "Drive output ZIP:",
        DRIVE_OUTPUT_ZIP,
    )

    print(
        "ZIP size (MB):",
        round(
            DRIVE_OUTPUT_ZIP.stat().st_size
            / 1e6,
            2,
        ),
    )

else:
    LOCAL_OUTPUT_ZIP = (
        Path("/mnt/data")
        / "NOTEBOOK13B_RECONCILED_FINAL_MANUSCRIPT_ASSETS.zip"
    )

    if LOCAL_OUTPUT_ZIP.exists():
        LOCAL_OUTPUT_ZIP.unlink()

    with zipfile.ZipFile(
        LOCAL_OUTPUT_ZIP,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as archive:
        for path in sorted(
            OUTPUT_ROOT.rglob("*")
        ):
            if path.is_file():
                archive.write(
                    path,
                    arcname=str(
                        path.relative_to(
                            OUTPUT_ROOT
                        )
                    ),
                )

    print(
        "Local output ZIP:",
        LOCAL_OUTPUT_ZIP,
    )

print(
    "\nRequired completion message:"
)

print(
    "Notebook 13B final validation: PASSED"
)

# After successful execution

The analysis is fully reconciled when the final cell prints:

```text
Notebook 13B final validation: PASSED
```

No further analytical notebook should be required. The next deliverables are:

1. Revised English manuscript for **AI—MDPI**.
2. English Supplementary Materials with editable equations.
3. Point-by-point response to the reviewer.

The manuscript must use Notebook 05 values for the primary champion metrics, Notebook 07 for selective classification, Notebook 08 v2 for canonical conformal prediction, and Notebook 12 v2 only for the reviewer-driven confirmatory analyses.